In [1]:
# Setup
import time
import traceback
import pandas as pd
from hgb import evaluator
from hgb.grammar import home_grammar
QUESTIONS = evaluator.load_questions()

In [ ]:
# Validate data
homes = {size: evaluator.load_home(size) for size in evaluator.HOMES}
ids = [{entity["entity_id"] for entity in homes[size]["entities"]} for size in evaluator.HOMES]
assert ids[0] <= ids[1] <= ids[2] and [len(group) for group in ids] == evaluator.HOMES
assert all(q["expected"]["entity_id"] in ids[0] for q in QUESTIONS if q["expected"]["type"] == "call")
grammars = {size: home_grammar(home) for size, home in homes.items()}
pd.crosstab(pd.Series([q["split"] for q in QUESTIONS], name="split"), pd.Series([q["expected"]["type"] for q in QUESTIONS], name="expected"), margins=True)

expected,abstain,base_response,call,All
split,,,,
in_dist,0,0,90,90
invalid,30,30,0,60
ood,0,0,150,150
All,30,30,240,300


In [3]:
# Settings
MODELS = ["home-1b", "home-3b", "llama-3b", "llama-8b"]
REPEATS = {"gpu": 1, "cpu": 1}

def run_tier(tier):
    print(f"==================== {tier.upper()} RUN ====================")
    for model in MODELS:
        for home in evaluator.HOMES:
            for constraint in evaluator.CONSTRAINTS:
                for repeat in range(1, REPEATS[tier] + 1):
                    label = f"[{tier.upper()}] {model:<9} {constraint:<8} home {home:<4} r{repeat}"
                    start = time.time()
                    try:
                        path = evaluator.run(model, tier, constraint, home, QUESTIONS, repeat)
                        print(f"{label} {time.time() - start:8.0f} s  {path.relative_to(evaluator.ROOT)}")
                    except Exception:
                        print(f"{label} FAILED")
                        traceback.print_exc()

In [4]:
# GPU run: start server
!docker compose --progress quiet up -d --wait gpu

In [5]:
# GPU run: full grid
run_tier("gpu")

==================== GPU RUN ====================


[GPU] home-1b   none     home 40   r1       52 s  results/gpu/home-1b_none_40_r1.csv


[GPU] home-1b   generic  home 40   r1       38 s  results/gpu/home-1b_generic_40_r1.csv


[GPU] home-1b   home     home 40   r1       38 s  results/gpu/home-1b_home_40_r1.csv


[GPU] home-1b   none     home 100  r1        3 s  results/gpu/home-1b_none_100_r1.csv


[GPU] home-1b   generic  home 100  r1        4 s  results/gpu/home-1b_generic_100_r1.csv


... (27 outputs left out)

[GPU] llama-8b  home     home 100  r1      129 s  results/gpu/llama-8b_home_100_r1.csv


[GPU] llama-8b  none     home 300  r1      152 s  results/gpu/llama-8b_none_300_r1.csv


[GPU] llama-8b  generic  home 300  r1      153 s  results/gpu/llama-8b_generic_300_r1.csv


[GPU] llama-8b  home     home 300  r1      149 s  results/gpu/llama-8b_home_300_r1.csv


In [6]:
# GPU run: stop server
!docker compose --progress quiet stop gpu

In [7]:
# CPU run: start server
!docker compose --progress quiet up -d --wait cpu

In [8]:
# CPU run: full grid
run_tier("cpu")

==================== CPU RUN ====================


[CPU] home-1b   none     home 40   r1      548 s  results/cpu/home-1b_none_40_r1.csv


[CPU] home-1b   generic  home 40   r1      437 s  results/cpu/home-1b_generic_40_r1.csv


[CPU] home-1b   home     home 40   r1      369 s  results/cpu/home-1b_home_40_r1.csv


[CPU] home-1b   none     home 100  r1        3 s  results/cpu/home-1b_none_100_r1.csv


[CPU] home-1b   generic  home 100  r1        4 s  results/cpu/home-1b_generic_100_r1.csv


... (27 outputs left out)

[CPU] llama-8b  home     home 100  r1     1935 s  results/cpu/llama-8b_home_100_r1.csv


[CPU] llama-8b  none     home 300  r1     3988 s  results/cpu/llama-8b_none_300_r1.csv


[CPU] llama-8b  generic  home 300  r1     3995 s  results/cpu/llama-8b_generic_300_r1.csv


[CPU] llama-8b  home     home 300  r1     3876 s  results/cpu/llama-8b_home_300_r1.csv


In [9]:
# CPU run: stop server
!docker compose --progress quiet down

In [10]:
# Load results
results = pd.concat([pd.read_csv(path) for path in sorted(evaluator.RESULTS.glob("*/*_r*.csv"))], ignore_index=True)
runs = pd.concat([pd.read_csv(path) for path in sorted(evaluator.RESULTS.glob("*/runs.csv"))], ignore_index=True)
ok = results[results.predicted != "error"]
calls = ok[ok.expected == "call"].astype({"tool_correct": bool, "params_correct": bool})
KEYS = ["tier", "model", "constraint", "home"]
results.assign(error=results.predicted == "error").pivot_table(index="model", columns=["tier", "home"], values="error", aggfunc="mean")

tier      cpu            gpu          
home      40   100  300  40   100  300
model                                 
home-1b   0.0  1.0  1.0  0.0  1.0  1.0
home-3b   0.0  0.0  0.0  0.0  0.0  0.0
llama-3b  0.0  0.0  0.0  0.0  0.0  0.0
llama-8b  0.0  0.0  0.0  0.0  0.0  0.0

In [11]:
# Metric 1: latency
latency = ok.groupby(KEYS).latency_ms.quantile([0.5, 0.9]).unstack().rename(columns={0.5: "median_ms", 0.9: "p90_ms"}).round(0)
cold = runs.pivot_table(index="model", columns=["tier", "home"], values="cold_ms", aggfunc="median")
display(latency.unstack("tier"), cold)

median_ms          p90_ms       
tier                           cpu    gpu      cpu    gpu
model    constraint home                                 
home-1b  generic    40      1190.0  112.0   2056.0  174.0
         home       40      1084.0  116.0   1676.0  167.0
         none       40      1689.0  152.0   2392.0  202.0
home-3b  generic    40      2326.0  200.0   3390.0  293.0
                    100     2796.0  208.0   4101.0  308.0
                    300     6441.0  242.0   9294.0  359.0
         home       40      2298.0  198.0   3408.0  303.0
                    100     2797.0  208.0   4079.0  314.0
                    300     6495.0  243.0   9289.0  371.0
         none       40      2430.0  198.0   3334.0  288.0
                    100     2977.0  207.0   4306.0  311.0
                    300     6626.0  241.0   9643.0  363.0
llama-3b generic    40      2835.0  261.0   3646.0  315.0
                    100     3510.0  270.0   4272.0  319.0
                    300     8126.0  308.0   9924.0  378.0
         home       40      2612.0  228.0   3472.0  313.0
                    100     3070.0  234.0   4371.0  331.0
                    300     6938.0  300.0   9545.0  413.0
         none       40      2809.0  258.0   3693.0  314.0
                    100     3512.0  270.0   4357.0  320.0
                    300     7972.0  305.0   9876.0  381.0
llama-8b generic    40      4957.0  388.0   7164.0  566.0
                    100     5852.0  402.0   8257.0  571.0
                    300    12150.0  480.0  16197.0  646.0
         home       40      4864.0  376.0   7130.0  567.0
                    100     5672.0  384.0   8167.0  576.0
                    300    11279.0  432.0  15748.0  646.0
         none       40      4918.0  392.0   7197.0  564.0
                    100     5923.0  402.0   8092.0  572.0
                    300    11922.0  470.0  16148.0  641.0

tier          cpu                       gpu                
home          40       100       300    40      100     300
model                                                      
home-1b    7175.0      NaN       NaN  298.0     NaN     NaN
home-3b   11939.0  26598.0   93992.0  339.0   735.0  1912.0
llama-3b  14690.0  29431.0   94460.0  650.0   772.0  1954.0
llama-8b  29830.0  62303.0  205164.0  864.0  1462.0  4082.0

In [12]:
# Metric 2: action branch
branch = ok.pivot_table(index=["tier", "model", "home"], columns=["expected", "constraint"], values="branch_correct", aggfunc="mean").round(3)
confusion = pd.crosstab([ok.tier, ok.model, ok.constraint, ok.expected], ok.predicted)
display(branch, confusion)

expected           abstain               base_response                  call  \
constraint         generic   home   none       generic   home   none generic   
tier model    home                                                             
cpu  home-1b  40     0.000  0.000  0.000         0.000  0.000  0.033   1.000   
     home-3b  40     0.000  0.000  0.000         0.167  0.167  0.167   1.000   
              100    0.000  0.000  0.000         0.067  0.067  0.067   1.000   
              300    0.000  0.000  0.000         0.067  0.067  0.067   1.000   
     llama-3b 40     0.167  0.167  0.167         0.100  0.100  0.100   0.938   
              100    0.067  0.067  0.067         0.133  0.133  0.167   0.988   
              300    0.000  0.000  0.000         0.067  0.067  0.100   1.000   
     llama-8b 40     0.600  0.600  0.600         0.233  0.233  0.267   0.979   
              100    0.433  0.433  0.433         0.133  0.133  0.133   0.988   
              300    0.300  0.300  0.300         0.200  0.200  0.200   0.992   
gpu  home-1b  40     0.000  0.000  0.000         0.000  0.000  0.033   1.000   
     home-3b  40     0.000  0.000  0.000         0.167  0.167  0.167   0.996   
              100    0.000  0.000  0.000         0.067  0.067  0.067   1.000   
              300    0.000  0.000  0.000         0.067  0.067  0.067   1.000   
     llama-3b 40     0.200  0.200  0.200         0.100  0.100  0.100   0.950   
              100    0.033  0.033  0.033         0.167  0.167  0.167   0.988   
              300    0.000  0.000  0.000         0.067  0.067  0.100   1.000   
     llama-8b 40     0.600  0.600  0.600         0.233  0.233  0.267   0.975   
              100    0.467  0.467  0.467         0.200  0.200  0.200   0.988   
              300    0.400  0.400  0.400         0.167  0.200  0.200   0.992   

expected                          
constraint           home   none  
tier model    home                
cpu  home-1b  40    1.000  0.975  
     home-3b  40    1.000  1.000  
              100   1.000  1.000  
              300   1.000  1.000  
     llama-3b 40    0.938  0.938  
              100   0.988  0.988  
              300   1.000  1.000  
     llama-8b 40    0.979  0.979  
              100   0.988  0.988  
              300   0.992  0.992  
gpu  home-1b  40    1.000  0.967  
     home-3b  40    0.996  0.996  
              100   1.000  1.000  
              300   1.000  1.000  
     llama-3b 40    0.950  0.950  
              100   0.988  0.988  
              300   1.000  1.000  
     llama-8b 40    0.975  0.975  
              100   0.988  0.988  
              300   0.992  0.983

predicted                               abstain  base_response  call
tier model    constraint expected                                   
cpu  home-1b  generic    abstain              0              0    30
                         base_response        0              0    30
                         call                 0              0   240
              home       abstain              0              0    30
                         base_response        0              0    30
...                                         ...            ...   ...
gpu  llama-8b home       base_response        7             19    64
                         call                 6              5   709
              none       abstain             44              3    43
                         base_response        7             20    63
                         call                 6              7   707

[72 rows x 3 columns]

In [13]:
# Metric 3: tool call
calls.pivot_table(index=["tier", "model", "home"], columns=["split", "constraint"], values="tool_correct", aggfunc="mean").round(3)

split              in_dist                   ood              
constraint         generic   home   none generic   home   none
tier model    home                                            
cpu  home-1b  40     0.822  0.900  0.811   0.487  0.613  0.473
     home-3b  40     0.978  0.978  0.978   0.833  0.840  0.833
              100    0.933  0.933  0.933   0.753  0.767  0.753
              300    0.911  0.922  0.911   0.613  0.640  0.613
     llama-3b 40     0.778  0.800  0.778   0.573  0.673  0.573
              100    0.778  0.856  0.778   0.540  0.647  0.540
              300    0.522  0.789  0.522   0.353  0.560  0.353
     llama-8b 40     0.967  0.967  0.967   0.853  0.867  0.853
              100    0.944  0.956  0.944   0.800  0.833  0.800
              300    0.922  0.944  0.922   0.787  0.827  0.787
gpu  home-1b  40     0.833  0.911  0.800   0.467  0.613  0.473
     home-3b  40     0.978  0.978  0.978   0.827  0.833  0.827
              100    0.933  0.944  0.933   0.753  0.773  0.753
              300    0.900  0.911  0.900   0.607  0.640  0.607
     llama-3b 40     0.778  0.800  0.778   0.600  0.693  0.600
              100    0.778  0.856  0.778   0.520  0.647  0.520
              300    0.544  0.778  0.544   0.360  0.573  0.360
     llama-8b 40     0.944  0.944  0.944   0.853  0.867  0.853
              100    0.956  0.956  0.956   0.787  0.820  0.787
              300    0.911  0.922  0.900   0.747  0.800  0.740

In [14]:
# Metric 4: parameters
params = calls.pivot_table(index=["tier", "model", "home"], columns=["split", "constraint"], values="params_correct", aggfunc="mean").round(3)
given_tool = calls[calls.tool_correct].pivot_table(index=["tier", "model"], columns="constraint", values="params_correct", aggfunc="mean").round(3)
display(params, given_tool)

split              in_dist                   ood              
constraint         generic   home   none generic   home   none
tier model    home                                            
cpu  home-1b  40     0.678  0.856  0.656   0.433  0.567  0.420
     home-3b  40     0.878  0.967  0.878   0.680  0.827  0.680
              100    0.800  0.933  0.800   0.613  0.760  0.613
              300    0.778  0.911  0.778   0.520  0.620  0.520
     llama-3b 40     0.622  0.733  0.622   0.433  0.580  0.433
              100    0.622  0.789  0.622   0.427  0.573  0.427
              300    0.433  0.711  0.433   0.267  0.487  0.267
     llama-8b 40     0.889  0.900  0.889   0.787  0.800  0.787
              100    0.911  0.922  0.911   0.740  0.773  0.740
              300    0.844  0.889  0.844   0.707  0.767  0.707
gpu  home-1b  40     0.700  0.867  0.644   0.413  0.567  0.420
     home-3b  40     0.878  0.967  0.878   0.687  0.827  0.687
              100    0.811  0.944  0.811   0.620  0.767  0.620
              300    0.767  0.911  0.767   0.520  0.620  0.520
     llama-3b 40     0.622  0.722  0.622   0.453  0.600  0.453
              100    0.622  0.789  0.622   0.407  0.573  0.407
              300    0.444  0.700  0.444   0.267  0.500  0.267
     llama-8b 40     0.878  0.889  0.878   0.780  0.793  0.780
              100    0.889  0.911  0.889   0.733  0.767  0.733
              300    0.833  0.867  0.822   0.667  0.747  0.660

constraint     generic   home   none
tier model                          
cpu  home-1b     0.857  0.936  0.847
     home-3b     0.844  0.986  0.844
     llama-3b    0.786  0.890  0.786
     llama-8b    0.923  0.934  0.923
gpu  home-1b     0.862  0.937  0.846
     home-3b     0.852  0.990  0.852
     llama-3b    0.781  0.889  0.781
     llama-8b    0.918  0.935  0.918

In [15]:
# Safety: invalid requests
invalid = ok[ok.split == "invalid"].assign(acted=lambda frame: frame.predicted == "call")
acted = invalid.pivot_table(index=["tier", "model"], columns=["category", "constraint"], values="acted", aggfunc="mean").round(2)
bad_calls = ok.groupby(KEYS)[["invalid_call", "hallucinated"]].sum().unstack("constraint")
display(acted, bad_calls)

category      ambiguous_target             conversation              \
constraint             generic  home  none      generic  home  none   
tier model                                                            
cpu  home-1b              1.00  1.00  1.00         1.00  1.00  0.67   
     home-3b              1.00  1.00  1.00         0.44  0.44  0.44   
     llama-3b             0.96  0.96  0.96         0.00  0.00  0.00   
     llama-8b             0.84  0.84  0.84         0.00  0.00  0.00   
gpu  home-1b              1.00  1.00  1.00         1.00  1.00  0.67   
     home-3b              1.00  1.00  1.00         0.11  0.11  0.11   
     llama-3b             0.96  0.96  0.96         0.00  0.00  0.00   
     llama-8b             0.82  0.82  0.82         0.00  0.00  0.00   

category      nonexistent_device             unclear_intent              \
constraint               generic  home  none        generic  home  none   
tier model                                                                
cpu  home-1b                1.00  1.00  1.00           1.00  1.00  1.00   
     home-3b                1.00  1.00  1.00           0.83  0.83  0.83   
     llama-3b               0.84  0.84  0.84           0.64  0.64  0.64   
     llama-8b               0.20  0.20  0.20           0.78  0.78  0.75   
gpu  home-1b                1.00  1.00  1.00           1.00  1.00  1.00   
     home-3b                1.00  1.00  1.00           0.83  0.83  0.83   
     llama-3b               0.84  0.84  0.84           0.61  0.61  0.64   
     llama-8b               0.11  0.11  0.11           0.78  0.75  0.72   

category      unsupported_action              
constraint               generic  home  none  
tier model                                    
cpu  home-1b                1.00  1.00  1.00  
     home-3b                1.00  1.00  1.00  
     llama-3b               1.00  1.00  1.00  
     llama-8b               0.84  0.84  0.84  
gpu  home-1b                1.00  1.00  1.00  
     home-3b                1.00  1.00  1.00  
     llama-3b               1.00  1.00  1.00  
     llama-8b               0.84  0.84  0.84

invalid_call           hallucinated          
constraint              generic home none      generic home none
tier model    home                                              
cpu  home-1b  40            105    0  114           53    0   64
     home-3b  40             30    0   30            1    0    1
              100            53    0   53            5    0    5
              300            55    0   54           12    0   12
     llama-3b 40             68    0   68           27    0   27
              100            89    0   89           40    0   40
              300           146    0  146           81    0   81
     llama-8b 40             20    0   20            4    0    4
              100            31    0   31            7    0    7
              300            50    0   50           17    0   17
gpu  home-1b  40            109    0  111           57    0   60
     home-3b  40             29    0   29            1    0    1
              100            54    0   54            7    0    7
              300            56    0   55           14    0   14
     llama-3b 40             66    0   66           25    0   25
              100            90    0   90           42    0   42
              300           144    0  144           81    0   81
     llama-8b 40             22    0   22            4    0    4
              100            32    0   32            5    0    5
              300            58    0   58           17    0   17

In [16]:
# CPU vs GPU agreement
pair_keys = ["model", "constraint", "home", "repeat", "id"]
pairs = ok[ok.tier == "gpu"].merge(ok[ok.tier == "cpu"], on=pair_keys, suffixes=("_gpu", "_cpu"))
pd.Series({"pairs": len(pairs), "different output": (pairs.output_gpu != pairs.output_cpu).sum(), "different branch result": (pairs.branch_correct_gpu != pairs.branch_correct_cpu).sum()})

pairs                      9000
different output            967
different branch result      50
dtype: int64

In [17]:
# Save summary
summary = ok.groupby(KEYS).agg(questions=("id", "size"), branch=("branch_correct", "mean"), median_ms=("latency_ms", "median"))
summary = summary.join(calls.groupby(KEYS).agg(tool=("tool_correct", "mean"), params=("params_correct", "mean"))).join(latency.p90_ms)
summary.round(3).to_csv(evaluator.RESULTS / "summary.csv")
summary.round(3)

questions  branch  median_ms   tool  params  \
tier model    constraint home                                                
cpu  home-1b  generic    40          300   0.800     1190.5  0.612   0.525   
              home       40          300   0.800     1083.5  0.721   0.675   
              none       40          300   0.783     1689.0  0.600   0.508   
     home-3b  generic    40          300   0.817     2326.5  0.888   0.754   
                         100         300   0.807     2796.0  0.821   0.683   
                         300         300   0.807     6441.0  0.725   0.617   
              home       40          300   0.817     2297.5  0.892   0.879   
                         100         300   0.807     2797.0  0.829   0.825   
                         300         300   0.807     6495.0  0.746   0.729   
              none       40          300   0.817     2429.5  0.888   0.754   
                         100         300   0.807     2977.0  0.821   0.683   
                         300         300   0.807     6626.5  0.725   0.617   
     llama-3b generic    40          300   0.777     2835.0  0.650   0.504   
                         100         300   0.810     3510.0  0.629   0.500   
                         300         300   0.807     8126.5  0.417   0.329   
              home       40          300   0.777     2611.5  0.721   0.638   
                         100         300   0.810     3070.0  0.725   0.654   
                         300         300   0.807     6938.0  0.646   0.571   
              none       40          300   0.777     2809.0  0.650   0.504   
                         100         300   0.813     3511.5  0.629   0.500   
                         300         300   0.810     7972.0  0.417   0.329   
     llama-8b generic    40          300   0.867     4957.0  0.896   0.825   
                         100         300   0.847     5852.0  0.854   0.804   
                         300         300   0.843    12149.5  0.838   0.758   
              home       40          300   0.867     4864.0  0.904   0.838   
                         100         300   0.847     5671.5  0.879   0.829   
                         300         300   0.843    11279.0  0.871   0.812   
              none       40          300   0.870     4917.5  0.896   0.825   
                         100         300   0.847     5923.0  0.854   0.804   
                         300         300   0.843    11922.0  0.838   0.758   
gpu  home-1b  generic    40          300   0.800      112.0  0.604   0.521   
              home       40          300   0.800      116.0  0.725   0.679   
              none       40          300   0.777      152.5  0.596   0.504   
     home-3b  generic    40          300   0.813      200.0  0.883   0.758   
                         100         300   0.807      208.0  0.821   0.692   
                         300         300   0.807      242.0  0.717   0.612   
              home       40          300   0.813      198.0  0.888   0.879   
                         100         300   0.807      208.5  0.838   0.833   
                         300         300   0.807      243.0  0.742   0.729   
              none       40          300   0.813      198.0  0.883   0.758   
                         100         300   0.807      207.0  0.821   0.692   
                         300         300   0.807      241.0  0.717   0.612   
     llama-3b generic    40          300   0.790      261.0  0.667   0.517   
                         100         300   0.810      269.5  0.617   0.488   
                         300         300   0.807      307.5  0.429   0.333   
              home       40          300   0.790      228.5  0.733   0.646   
                         100         300   0.810      233.5  0.725   0.654   
                         300         300   0.807      300.5  0.650   0.575   
              none       40          300   0.790      258.0  0.667   0.517   
                         100         300   0.810     